# 01 — ADME EDA (Part 1) - Biogen Dataset

**Goal**: Understand the ADME dataset thoroughly before making any cleaning or modelling decisions.  
**Dataset**: `data/raw/ADME_public_set_3521.csv` — 3521 compounds, 6 log-transformed ADME endpoints.  
**Outputs**: Findings documented in Section 1.10; no data modified here.

> **Archived**: early-stage EDA notebook, kept for a few analyses not repeated elsewhere (Lipinski Ro5 violations, pairwise-similarity walkthrough, missingness correlation, floor-value spike investigation). The baseline-model and hyperparameter-tuning sections that originally followed this EDA have been removed — they used ECFP4/RDKit2D features and predate the paper-faithful methodology; see `01.5_adme_biogen_public_recreation.ipynb` and `01.6_adme_paper_recreation_results.ipynb` for the current modelling pipeline.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.eda import smiles_validity_report, missing_value_report, max_corr_report
from src.features import rdkit_descriptors
from src.plotting import endpoint_distributions

SEED = 42
DATA_PATH = '../data/raw/ADME_public_set_3521.csv'

TRAIN_MPNN2 = True        # set True to train MPNN2 (graph + rdkit_2d_normalized)
TRAIN_MPNN_GRAPH = True   # set True to train MPNN (graph-only, no rdkit descriptors)
TUNE_MPNN2 = True # set True to run grid search (~30-60 min)
RUN_RADIUS_SENSITIVITY = False  # set True to run 2.4b radius sweep (~2 min)

ENDPOINT_COLS = [
    'LOG HLM_CLint (mL/min/kg)',
    'LOG MDR1-MDCK ER (B-A/A-B)',
    'LOG SOLUBILITY PH 6.8 (ug/mL)',
    'LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound)',
    'LOG PLASMA PROTEIN BINDING (RAT) (% unbound)',
    'LOG RLM_CLint (mL/min/kg)',
]
EP_SHORT = ['HLM', 'MDR1', 'SOL', 'PPB_H', 'PPB_R', 'RLM']
EP_SHORT_MAP = dict(zip(ENDPOINT_COLS, EP_SHORT))

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Imports OK')

## 1.1 — Load & Inspect

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nDtypes:')
print(df.dtypes)
df.head()

## 1.2 — SMILES Validity Report

Report only — no rows dropped here. Dropping is a cleaning decision made after EDA.

In [ ]:
validity = smiles_validity_report(df, smiles_col='SMILES')
print(f"Valid SMILES  : {validity['valid_count']} ({validity['valid_count']/len(df)*100:.2f}%)")
print(f"Invalid SMILES: {validity['invalid_count']} ({validity['invalid_count']/len(df)*100:.2f}%)")
if validity['invalid_indices']:
    print(f"Invalid row indices: {validity['invalid_indices']}")
    print(df.loc[validity['invalid_indices'], ['Internal ID', 'SMILES']])
else:
    print('No invalid SMILES found.')

## 1.3 — Duplicate Check

In [ ]:
from rdkit import Chem

# Duplicate raw SMILES strings
dup_raw = df['SMILES'].duplicated(keep=False)
print(f"Duplicate raw SMILES: {dup_raw.sum()} rows ({df['SMILES'].duplicated().sum()} extra copies)")

# Canonical SMILES duplicates (catches representation variants)
def to_canonical(smi):
    if pd.isna(smi):
        return None
    mol = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(mol) if mol is not None else None

df['canonical_smiles'] = df['SMILES'].apply(to_canonical)
dup_can = df['canonical_smiles'].duplicated(keep=False) & df['canonical_smiles'].notna()
n_dup_can = df['canonical_smiles'].duplicated().sum()
print(f"Duplicate canonical SMILES: {dup_can.sum()} rows ({n_dup_can} extra copies)")

if n_dup_can > 0:
    dup_df = df[dup_can].sort_values('canonical_smiles')
    # Check endpoint consistency among duplicates
    dup_ep_std = dup_df.groupby('canonical_smiles')[ENDPOINT_COLS].std()
    inconsistent = (dup_ep_std > 0).any(axis=1)
    print(f"Duplicates with inconsistent endpoint values: {inconsistent.sum()}")
    display(dup_df[['Internal ID', 'canonical_smiles'] + ENDPOINT_COLS].head(20))

SMILES = string encoding of molecule
Canoncial SMILES = single, deterministic string generated via an algo (RDKit), always produces the same SMILES regardless of how molecule originally encoded.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

# ── 1. Show the transformation on synthetic examples ──────────────────────────
non_canonical_examples = [
    ('Ethanol (reversed)', 'OCC'),
    ('Aspirin (variant A)',  'OC(=O)c1ccccc1OC(C)=O'),
    ('Aspirin (variant B)',  'CC(=O)Oc1ccccc1C(O)=O'),
]

rows = []
for name, smi in non_canonical_examples:
    mol = Chem.MolFromSmiles(smi)
    canon = Chem.MolToSmiles(mol)
    rows.append({'Name': name, 'Input SMILES': smi, 'Canonical SMILES': canon,
                 'Changed?': '✓' if smi != canon else '—'})

print("Synthetic examples — SMILES → Canonical:")
display(pd.DataFrame(rows))

# ── 2. Draw aspirin variants side-by-side to show same structure ──────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, (name, smi) in zip(axes, non_canonical_examples[1:]):
    mol = Chem.MolFromSmiles(smi)
    canon = Chem.MolToSmiles(mol)
    img = Draw.MolToImage(mol, size=(320, 220))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Input:  {smi}\n→ Canon: {canon}', fontsize=8)
plt.suptitle('Same molecule (aspirin), two input strings → identical canonical SMILES', fontsize=10)
plt.tight_layout()
plt.show()

# ── 3. Verify our dataset: original vs canonical for first 5 compounds ─────────
print("\nDataset spot-check — first 5 compounds:")
check = df[['Internal ID', 'SMILES', 'canonical_smiles']].head()
check['changed'] = check['SMILES'] != check['canonical_smiles']
display(check)

## 1.4 — Missing Value Report

In [ ]:
import os
os.makedirs('../figures', exist_ok=True)

miss_report = missing_value_report(df, ENDPOINT_COLS)
print('Missing values per endpoint:')
display(miss_report)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
miss_report['pct_missing'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('% Missing per Endpoint')
ax.set_ylabel('% Missing')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../figures/1.2_missing_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# Heatmap of missingness pattern
miss_matrix = df[ENDPOINT_COLS].isna().astype(int)
miss_matrix.columns = EP_SHORT

fig, ax = plt.subplots(figsize=(8, 3))
miss_corr = miss_matrix.corr()
sns.heatmap(miss_corr, annot=True, fmt='.2f', cmap='Oranges', ax=ax, vmin=0, vmax=1)
ax.set_title('Missingness Co-occurrence (correlation of NaN indicators)')
plt.tight_layout()
plt.savefig('../figures/1.2_missingness_cooccurrence.png', dpi=150, bbox_inches='tight')
plt.show()

HLM/RLM correlated in missingness -> Not all compounds tested so if a compound wasnt sent to the lab, its likely to be missing both not one or the other?

PPB_ H/R Same story, correlated as generally tested together?

#### 1.4b — Maximum Possible Correlation (WIP)

Upper bound on the Pearson r any model could achieve for each endpoint, given the assay's measurement noise.  
Uses the Brown, Muchmore & Hajduk simulation: adds Gaussian noise of magnitude `error` to the observed values and correlates against the originals over 1000 iterations.  
A model's r can be interpreted relative to this ceiling.

In [ ]:
mc_report = max_corr_report(df, ENDPOINT_COLS)
mc_report.index = EP_SHORT
print('Maximum possible R² per endpoint at 2x, 3x, 5x, 10x assay noise (Brown, Muchmore & Hajduk):')
display(mc_report.round(3))

## 1.5 — Summary Statistics

In [ ]:
desc = df[ENDPOINT_COLS].describe().T

skew = df[ENDPOINT_COLS].apply(lambda x: stats.skew(x.dropna()))
kurt = df[ENDPOINT_COLS].apply(lambda x: stats.kurtosis(x.dropna()))

desc['skewness'] = skew
desc['kurtosis'] = kurt
desc.index = EP_SHORT

print('Summary statistics (all 6 endpoints):')
display(desc[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']])

display(desc[['skewness', 'kurtosis']])

SOL only column with | skew | > 1

## 1.6 — Outlier Detection

Flag compounds >3σ from the mean per endpoint. Report only — no rows removed.

In [ ]:
outlier_summary = {}
outlier_indices = set()

for col in ENDPOINT_COLS:
    col_data = df[col].dropna()
    mu, sigma = col_data.mean(), col_data.std()
    mask = (df[col] - mu).abs() > 3 * sigma
    flagged = df.index[mask & df[col].notna()].tolist()
    outlier_summary[col] = len(flagged)
    outlier_indices.update(flagged)

print('Outlier counts per endpoint (>3σ):')
for k, v in outlier_summary.items():
    print(f"  {k}: {v}")
print(f"\nUnique compounds flagged in any endpoint: {len(outlier_indices)}")

if outlier_indices:
    print('\nFlagged rows (first 20):')
    display(df.loc[sorted(outlier_indices)[:20], ['Internal ID', 'SMILES'] + ENDPOINT_COLS])

Consider IQR outlier detection, better for skewed data

## 1.7 — Endpoint Distributions

In [ ]:
fig = endpoint_distributions(df, ENDPOINT_COLS)
plt.savefig('../figures/1.3_endpoint_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

**Distribution notes**:
- HLM_CLint: +ve skew, tail points right
- MDR1-MDCK ER: +ve skew
- SOLUBILITY: -ve skew
- PPB HUMAN: mild -ve skew
- PPB RAT: mild -ve skew
- RLM_CLint: No strong peak

**Floor-value spike in HLM_CLint and RLM_CLint**

The first bin spike visible in HLM_CLint and RLM_CLint is not a single outlier — it is a large cluster of compounds sharing the exact same minimum value:

| Endpoint | Floor value (log) | Compounds at floor | % of endpoint data |
|---|---|---|---|
| HLM_CLint | 0.676 | 958 | 31.0% |
| RLM_CLint | 1.028 | 346 | 11.3% |

The cause is currently unknown — possibilities include a lower limit of quantification when the measurements were taken in the lab.

**Why IQR does not flag it**: for HLM, Q1 equals the floor value (0.676) because >25% of the data sits there. The IQR lower bound is therefore −1.015 — well below any observed value — so no compound is flagged.

**Decision**: keep floor values as-is. All models see the same systematic pattern, so relative comparisons remain valid. The practical effect is that models partially learn to predict the floor for a large fraction of compounds, which may inflate apparent accuracy near that value.

## 1.8 — Endpoint Correlations

In [ ]:
corr = df[ENDPOINT_COLS].corr(method='spearman')
corr.index = corr.columns = EP_SHORT

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1, ax=ax, square=True
)
ax.set_title('Spearman Correlation — Endpoints')
plt.tight_layout()
plt.savefig('../figures/1.8_endpoint_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.9 — Chemical Space Overview

In [ ]:
# Pre-validate SMILES before featurising — rdkit_descriptors raises on invalid input
assert validity["invalid_count"] == 0, f"Fix {validity['invalid_count']} invalid SMILES before featurising"

# Compute RDKit 2D descriptors (uses only valid SMILES)
desc_df = rdkit_descriptors(df['SMILES'].tolist())
desc_df.index = df.index
print('Descriptor shape:', desc_df.shape)
display(desc_df.describe())

MW = Molecular weight
LogP = Lipophilicity
TPSA = Topological polar surface area - absorption/permeability
HBD = H-bond donors
HBA = H-bond acceptors
RotBonds = Rotatable single bonds - flexibility

lipophiliicty - how fat loving molec is
tpa - tracks with molec weight§§

In [ ]:
# Histograms of physicochemical properties
prop_labels = {
    'MW': 'Molecular Weight (Da)',
    'LogP': 'LogP',
    'TPSA': 'TPSA (Å²)',
    'HBD': 'H-Bond Donors',
    'HBA': 'H-Bond Acceptors',
    'RotBonds': 'Rotatable Bonds',
}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, (col, label) in zip(axes.flatten(), prop_labels.items()):
    data = desc_df[col].dropna()
    ax.hist(data, bins=40, color='teal', edgecolor='white', alpha=0.8)
    ax.set_title(label)
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
plt.tight_layout()
plt.suptitle('Physicochemical Property Distributions', y=1.02)
plt.savefig('../figures/1.5_physicochemical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Lipinski Ro5 violations (informational only)
lip_violations = (
    (desc_df['MW'] > 500).astype(int) +
    (desc_df['LogP'] > 5).astype(int) +
    (desc_df['HBD'] > 5).astype(int) +
    (desc_df['HBA'] > 10).astype(int)
)

print('Lipinski Ro5 violations per compound:')
print(lip_violations.value_counts().sort_index())
print(f"\nCompounds with >=2 violations (Ro5 fails): {(lip_violations >= 2).sum()} "
      f"({(lip_violations >= 2).mean()*100:.1f}%)")

heuristic, analysis of all drugs FDA approved, found trend that 5 rules that generally observable in drugs approved

3332 compounds break 0 rules, 148 break 1 rule etc

## 1.9b — Pairwise Chemical Similarity Distribution

How diverse is the compound library? If most compounds are very similar (high Tanimoto), the dataset is clustered around a few scaffolds; if similarity is low, the library covers a broad chemical space.

**Method**: Tanimoto similarity on ECFP4 Morgan fingerprints (radius=2, 2048 bits). For N compounds, there are N×(N−1)/2 unique pairs. RDKit's `BulkTanimotoSimilarity` computes all similarities of one fingerprint against a list in a single C++ call — much faster than a Python double loop.

**How Tanimoto works**: Each molecule is represented as a binary fingerprint (bit vector). The Tanimoto coefficient between two fingerprints A and B is:

$$T(A,B) = \frac{|A \cap B|}{|A \cup B|} = \frac{\text{bits on in both}}{\text{bits on in either}}$$

- T = 1.0 → identical fingerprints (same substructure features)
- T = 0.0 → no shared bits (completely different features)
- T ≈ 0.3–0.5 → typical for unrelated drug-like molecules

In [ ]:
# ── Small subsample demo: how BulkTanimotoSimilarity works ───────────────────
from rdkit.Chem import AllChem
from rdkit import Chem, DataStructs

# Pick 5 molecules to illustrate
demo_smiles = df['SMILES'].iloc[:5].tolist()
demo_fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), radius=2, nBits=2048)
            for s in demo_smiles]

# BulkTanimotoSimilarity: compute similarity of fp[0] against ALL others in one C++ call
sims_to_first = DataStructs.BulkTanimotoSimilarity(demo_fps[0], demo_fps[1:])
print('Tanimoto similarities of compound 0 vs compounds 1–4:')
for i, sim in enumerate(sims_to_first, start=1):
    print(f'  0 vs {i}: {sim:.3f}')

# For all unique pairs (upper triangle), we loop i and use Bulk for each row
print(f'\nAll pairwise similarities (5 compounds, {5*4//2} unique pairs):')
for i in range(len(demo_fps)):
    sims = DataStructs.BulkTanimotoSimilarity(demo_fps[i], demo_fps[i+1:])
    for j, sim in enumerate(sims, start=i+1):
        print(f'  {i} vs {j}: {sim:.3f}')

In [ ]:
# ── Full dataset pairwise similarity distribution ────────────────────────────
# 3521 compounds → 3521×3520/2 = 6,196,960 unique pairs

all_smiles = df['SMILES'].tolist()
all_fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), radius=2, nBits=2048)
           for s in all_smiles]
print(f'Computed {len(all_fps)} fingerprints')

# Collect all pairwise Tanimoto similarities using BulkTanimotoSimilarity
all_sims = []
for i in range(len(all_fps)):
    sims = DataStructs.BulkTanimotoSimilarity(all_fps[i], all_fps[i+1:])
    all_sims.extend(sims)

all_sims = np.array(all_sims)
print(f'Pairwise similarities: {len(all_sims):,} pairs')
print(f'Mean: {all_sims.mean():.3f}, Median: {np.median(all_sims):.3f}, '
      f'Std: {all_sims.std():.3f}')
print(f'Min: {all_sims.min():.3f}, Max: {all_sims.max():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(all_sims, bins=100, color='steelblue', alpha=0.8, edgecolor='none')
axes[0].axvline(all_sims.mean(), color='red', linestyle='--', linewidth=1.5,
                label=f'Mean = {all_sims.mean():.3f}')
axes[0].axvline(np.median(all_sims), color='orange', linestyle='--', linewidth=1.5,
                label=f'Median = {np.median(all_sims):.3f}')
axes[0].set_xlabel('Tanimoto Similarity (ECFP4)')
axes[0].set_ylabel('Number of Pairs')
axes[0].set_title('Pairwise Tanimoto Similarity Distribution')
axes[0].legend()
axes[0].set_xlim(0, 1)

# Cumulative distribution
sorted_sims = np.sort(all_sims)
cdf = np.arange(1, len(sorted_sims) + 1) / len(sorted_sims)
axes[1].plot(sorted_sims, cdf, color='steelblue', linewidth=1.5)
axes[1].axhline(0.5, color='grey', linestyle=':', alpha=0.5)
axes[1].axvline(np.median(all_sims), color='orange', linestyle='--', linewidth=1,
                label=f'Median = {np.median(all_sims):.3f}')
axes[1].set_xlabel('Tanimoto Similarity (ECFP4)')
axes[1].set_ylabel('Cumulative Fraction of Pairs')
axes[1].set_title('CDF of Pairwise Similarity')
axes[1].legend()
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.savefig('../figures/1.6_pairwise_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary statistics by threshold
for thresh in [0.3, 0.5, 0.7, 0.85]:
    frac = (all_sims >= thresh).mean() * 100
    print(f'Pairs with Tanimoto >= {thresh}: {frac:.1f}%')

## 1.10 — EDA Conclusions

> Findings are documented; cleaning strategy (SYNC-002) decided post-EDA review.

### Invalid SMILES
- **Finding**: 0 invalid SMILES out of 3521 (100% valid). No action required.

### Duplicates
- **Finding**: 0 duplicate raw SMILES; 0 duplicate canonical SMILES. Dataset is clean on this front.

### Missingness
- **Finding**: Missingness varies dramatically by endpoint:
  - HLM: 434 missing (12.3%), RLM: 467 missing (13.3%) — manageable
  - MDR1: 879 missing (25.0%), SOL: 1348 missing (38.3%) — substantial
  - PPB_HUMAN: 3327 missing (94.5%), PPB_RAT: 3353 missing (95.2%) — near-complete missingness
- **Cleaning strategy to decide (SYNC-002)**: PPB endpoints are nearly empty. Per-endpoint filtering (train each model on rows with non-missing values for that endpoint) is strongly preferred over complete-cases (which would leave only ~5% of data).

### Endpoint Distributions & Outliers
- **Skewness**: SOLUBILITY is left-skewed (skew=-1.64); HLM and MDR1 are mildly right-skewed (~0.6–0.8); RLM, PPB endpoints near-symmetric.
- **Outliers (>3σ)**: SOLUBILITY has 20 flagged outliers — largest concern. HLM: 3, MDR1: 1, PPB_RAT: 1. ~25 unique compounds flagged across all endpoints.
- **Endpoints to watch**: SOLUBILITY (high skew + most outliers + high missingness).

### Chemical Space
- **Drug-likeness**: 94.7% of compounds have 0 Lipinski Ro5 violations.
- **Violations**: 148 compounds have 1 violation, 41 have ≥2 (Ro5 fails).

## 1.10a — Stereoisomer Exclusion Check

Check for stereoisomer pairs with >3-fold difference in any log-scale endpoint. Following the filtering criterion in Fang et al. (ADME prospective validation paper), we flag and exclude any such pairs found.

**Why this matters**: Our ECFP4 fingerprints are computed with `useChirality=False` (the standard default), so the Morgan algorithm ignores stereochemistry annotations (R/S, E/Z). Two stereoisomers share the same atom connectivity and differ only in 3D spatial arrangement, producing **identical fingerprint bit vectors**. If they also differ substantially in activity, the model sees the same input for different target values — irresolvable label noise.

> **Why not set `useChirality=True`?** Stereochemistry annotations in SMILES are often incomplete or inconsistent across datasets; encoding unreliable chirality can introduce more noise than it resolves. The chirality-off default is standard practice in QSAR fingerprinting for this reason.

In [ ]:
from src.cleaning import exclude_stereoisomer_pairs

n_before = len(df)
df, excluded_stereo = exclude_stereoisomer_pairs(df, 'SMILES', ENDPOINT_COLS, fold_threshold=3)
n_after = len(df)

print(f'Stereoisomer exclusion (>3-fold difference in any log-scale endpoint):')
print(f'  Compounds before: {n_before}')
print(f'  Excluded:         {n_before - n_after}  (indices: {excluded_stereo})')
print(f'  Compounds after:  {n_after}')

## 1.11 — IQR Outlier Detection

Flag compounds outside [Q1 − 1.5·IQR, Q3 + 1.5·IQR] per endpoint.  
**Policy**: flag only — no rows removed. Outliers are kept for now, can look into how their exclusion later affects performance.  
Note: 3σ detection was used in Section 1.6 for reference; IQR is preferred here as it is more robust for skewed distributions (SOLUBILITY skewness = −1.64).

In [ ]:
from src.cleaning import flag_iqr_outliers, filter_endpoint

iqr_outlier_summary = {}

for col, short in zip(ENDPOINT_COLS, EP_SHORT):
    tag_col = f'iqr_outlier_{short}'
    df[tag_col] = False
    df_ep = filter_endpoint(df, col)
    mask = flag_iqr_outliers(df_ep, col)
    df.loc[mask[mask].index, tag_col] = True
    iqr_outlier_summary[col] = int(mask.sum())

print('IQR outlier counts per endpoint (1.5×IQR rule, flag only):')
for col, n in iqr_outlier_summary.items():
    print(f'  {col}: {n}')

print('\nOutlier tag columns added to df:', [f'iqr_outlier_{s}' for s in EP_SHORT])
print('\nExample — SOL outliers:')
display(df[df['iqr_outlier_SOL']][['Internal ID', 'SMILES', 'LOG SOLUBILITY PH 6.8 (ug/mL)']].head())

## 1.12 — Per-Endpoint Row Counts

Effective N for each model under per-endpoint filtering.  
PPB models (~170–190 rows) will have wider confidence intervals than HLM/RLM models (~3000 rows).

In [ ]:
from src.cleaning import filter_endpoint

header = f"{'Endpoint':<52} {'N_total':>7} {'N_missing':>10} {'N_for_model':>13} {'IQR_outliers':>13}"
print(header)
print('-' * len(header))

for col, short in zip(ENDPOINT_COLS, EP_SHORT):
    n_missing = int(df[col].isna().sum())
    n_model = len(filter_endpoint(df, col))
    n_iqr = iqr_outlier_summary[col]
    print(f'  {short:<50} {len(df):>7} {n_missing:>10} {n_model:>13} {n_iqr:>13}')